# 25 · Hard-instance frontier search for the single-cluster QUBO benchmark

The current QUBO target `long_wide_window_seed_7602` is **hard to prove optimal in the full MILP, but not hard for the problem the QUBO actually encodes**: the energy-only proxy MILP (effective prices + GPU capacity + one-hot) solves it to optimality in about 6 s with an LP-relaxation gap of about 0.5 %. Most of the full-MILP difficulty comes from terms the QUBO drops (renewable/grid dispatch and peak billing).

This notebook searches for single-cluster, 15-minute instances that are harder **on the proxy MILP**, while staying inside the QUBO size budget (≤ 1,000 assignment variables), and exports them in exactly the format notebook 21 loads.

Workflow:

1. Setup and constants.
2. Baseline sanity check: the extended generator with default flags must reproduce seed 7602, and the hardness predictors must match the measured values.
3. Solve function: proxy MILP and full MILP with timing, incumbent, bound, gap, node count and LP relaxation.
4. Outcome labels and hardness score, applied to the baseline.
5. Stage 1a: screening grid over utilisation × start-window width × node-size set.
6. Stage 1b: one-factor-at-a-time changes to price shape, power intensity, steering and planting.
7. Stage 2: confirmation of the top profiles on fresh seeds with a longer time limit.
8. Analysis plots and predictor correlations.
9. Export of confirmed profiles (2 development seeds + 1 held-out target).

**Ranking is always done on the proxy MILP.** The full MILP is solved in Stage 2 (and for the baseline) so the exported `gurobi_summary.csv` keeps the columns notebook 21 and `solve_one` use.

Set the environment variable `HARD_FRONTIER_SMOKE=1` before starting the kernel to run a reduced end-to-end smoke test (tiny grid, short time limits, separate output folders).

## 1. Setup

Imports, paths and run constants. Every Gurobi result row logs the Gurobi version, time limit, `MIPGap` target and thread count.

Long stages are opt-in through `RUN_*` flags. All solves are cached per instance: a folder that already contains `gurobi_summary.csv` is read instead of re-solved, and a recorded generation failure is not retried. Delete a stage folder under `results/hard_frontier/instances/` to force a re-run.

In [ ]:
from __future__ import annotations

import json
import math
import os
import shutil
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPOSITORY_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'src').is_dir() and (folder / 'pyproject.toml').exists()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError('Start the kernel inside the data_center_scheduling repository.')
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.config import ModelConfig
from src.data.scenarios import ENERGY_SCENARIOS
from src.instance_generator.hardness_metrics import compute_hardness_metrics, lp_root_gap
from src.instance_generator.price_shapes import apply_price_jitter, apply_two_valleys, build_energy_for_day
from src.instance_generator.single_cluster_stress import (
    SingleClusterStressConfig,
    generate_single_cluster_stress_instance,
    jobs_for_target_utilization,
)
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model
from src.quantum.benchmarking import make_effective_price_qubo_inputs

In [ ]:
SMOKE_TEST = os.environ.get('HARD_FRONTIER_SMOKE', '0') == '1'

# Solver settings (spec section 6.1).
TIME_LIMIT_SCREEN = 60
TIME_LIMIT_CONFIRM = 300
MIP_GAP = 1e-4
THREADS = min(8, os.cpu_count() or 1)

# Outcome labels and budgets.
HARD_GAP_THRESHOLD = 0.005          # optimality_hard if gap > 0.5 % at the time limit
ASSIGNMENT_BUDGET = 1_000           # QUBO assignment bits per exported instance
MIN_HARD_SEEDS = 3                  # Stage 2: keep a profile if >= 3 of 5 seeds are hard

# Generator settings shared by every profile.
UTILIZATION_TOLERANCE = 0.02
MAX_GENERATION_ATTEMPTS = 800
DURATION_VALUES = (4, 8, 12, 16)
DURATION_PROBABILITIES = (0.15, 0.35, 0.35, 0.15)
SIZE_SETS = {
    'A': ((16, 32, 64, 96), (0.15, 0.35, 0.35, 0.15)),   # baseline
    'B': ((48, 80, 112), (0.40, 0.40, 0.20)),            # non-dividing demands
    'C': ((32, 64, 96, 128), (0.25, 0.35, 0.25, 0.15)),  # lumpier demands
}
UTILIZATION_LEVELS = (0.82, 0.88, 0.94)
WINDOW_LEVELS = ((8, 16), (12, 20), (20, 28))

# Price-shape parameters (Stage 1b).
TWO_VALLEY_NIGHT_SLOTS = tuple(range(12, 24))   # 03:00-06:00
TWO_VALLEY_DEPTH = 0.10                          # lower night prices to the 10 % price quantile
JITTER_REL_STD = 0.01

# Full-MILP settings, identical to solve_one in scripts/run_long_duration_focused_sweep.py.
CONTRACTED_POWER_RATIO = 0.70
PEAK_PRICE = 1_000.0
DELTA_T = 0.25

SCREEN_SEEDS = (7701, 7702, 7703)
CONFIRM_SEEDS = (7801, 7802, 7803, 7804, 7805)
TOP_STAGE1A_FOR_STAGE1B = 3
TOP_FOR_CONFIRMATION = 5

RUN_SCREENING = True
RUN_PRICE_STRUCTURE = True
RUN_CONFIRMATION = False      # about 5 profiles x 5 seeds x (300 s + 300 s) in the worst case

RESULTS_ROOT = REPOSITORY_ROOT / 'results/hard_frontier'
EXPORT_ROOT = REPOSITORY_ROOT / 'data/instances/single_cluster_15m/hard_frontier'
BASELINE_DIR = REPOSITORY_ROOT / 'data/instances/single_cluster_15m/long_duration_focused/long_wide_window_seed_7602'

if SMOKE_TEST:
    TIME_LIMIT_SCREEN, TIME_LIMIT_CONFIRM = 10, 15
    UTILIZATION_LEVELS = (0.82,)
    WINDOW_LEVELS = ((12, 20),)
    SIZE_SETS = {key: SIZE_SETS[key] for key in ('A', 'B')}
    SCREEN_SEEDS, CONFIRM_SEEDS = (7701,), (7801, 7802, 7803)
    TOP_STAGE1A_FOR_STAGE1B, TOP_FOR_CONFIRMATION = 1, 1
    MIN_HARD_SEEDS = 0          # exercise the export path even when nothing is hard
    RUN_CONFIRMATION = True
    RESULTS_ROOT = REPOSITORY_ROOT / 'results/hard_frontier_smoke'
    EXPORT_ROOT = RESULTS_ROOT / 'export'

FIGURE_ROOT = RESULTS_ROOT / 'figures'
for folder in (RESULTS_ROOT, FIGURE_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

GUROBI_VERSION = '.'.join(str(part) for part in gp.gurobi.version())
SOLVER_LOG = {
    'gurobi_version': GUROBI_VERSION,
    'threads': THREADS,
    'mip_gap_target': MIP_GAP,
}
print(f'Gurobi {GUROBI_VERSION}; threads={THREADS}; MIPGap={MIP_GAP}; '
      f'screen={TIME_LIMIT_SCREEN}s; confirm={TIME_LIMIT_CONFIRM}s; smoke={SMOKE_TEST}')
print(f'Results: {RESULTS_ROOT.relative_to(REPOSITORY_ROOT)}; export: {EXPORT_ROOT.relative_to(REPOSITORY_ROOT)}')

# The size-limited pip licence fails above 2,000 variables or constraints.
_licence_probe = gp.Model()
_licence_probe.Params.OutputFlag = 0
_licence_probe.addVars(2_500)
_licence_probe.optimize()
print('Full Gurobi licence available.')

## 2. Baseline sanity check

Regenerate `long_wide_window` seed 7602 with the extended generator, leaving every new flag at its default. The output must equal the stored `jobs.csv` (the unit test `test_default_generation_is_byte_identical_to_reference` checks the same thing with SHA-256 hashes).

Then compute the structural predictors and compare them with the values measured before this search (tolerance 0.01).

In [ ]:
BASELINE_CONFIG = SingleClusterStressConfig(
    num_jobs=42,
    target_utilization=0.82,
    utilization_tolerance=0.06,
    node_demand_values=(16, 32, 64, 96),
    node_demand_probabilities=(0.15, 0.35, 0.35, 0.15),
    duration_values=(4, 8, 12, 16),
    duration_probabilities=(0.15, 0.35, 0.35, 0.15),
    min_start_options=12,
    max_start_options=20,
    min_price_spread_ratio=1.15,
    min_peak_potential_load_ratio=1.50,
    max_attempts=100,
)
base_energy = build_energy_for_day('base')
(baseline_jobs, baseline_clusters, baseline_energy,
 baseline_hidden, _, baseline_metadata) = generate_single_cluster_stress_instance(
    base_energy, BASELINE_CONFIG, 7602, 'long_wide_window_seed_7602'
)

if (BASELINE_DIR / 'jobs.csv').exists():
    pd.testing.assert_frame_equal(pd.read_csv(BASELINE_DIR / 'jobs.csv'), baseline_jobs, check_dtype=False)
    print('Regenerated seed 7602 equals the stored jobs.csv.')
else:
    print(f'Stored instance not found at {BASELINE_DIR}; relying on the hash regression test instead.')

baseline_metrics = compute_hardness_metrics(baseline_jobs, baseline_energy, baseline_clusters)
expected_baseline = {
    'utilization': 0.82, 'omega': 16, 'max_rho': 3.69, 'slots_rho_gt_1': 91,
    'rho_at_cheapest': 2.94, 'cheapest_slot': 48, 'distinct_job_types': 14,
    'savings_weight_corr': 0.68,
}
check = pd.DataFrame([
    {'metric': key, 'expected': value, 'measured': baseline_metrics[key],
     'ok': abs(baseline_metrics[key] - value) <= 0.01}
    for key, value in expected_baseline.items()
])
display(check)
assert check['ok'].all(), 'baseline predictors drifted from the measured reference values'
display(pd.DataFrame([baseline_metrics]))

## 3. Solve function

`solve_instance` builds both models with the repository's `build_milp_model` and solves them with `solve_model`:

- **Proxy MILP**: the energy-only model that uses exactly the QUBO's effective prices (`make_effective_price_qubo_inputs`), GPU capacity and one-hot assignment. `proxy_model_config` is copied from notebook 21. This is the model used for ranking.
- **Full MILP**: as in `solve_one`, with contracted power at 70 % of facility capacity and a peak price of 1,000.

For each model we record status, whether an incumbent exists, runtime, time to the first incumbent (a `MIPSOL` callback), objective, best bound, gap, node count and the LP-relaxation objective (`model.relax()`, solved before the MIP). `solve_model` is called with `raise_on_no_solution=False` so a time limit without an incumbent is recorded as `feasibility_hard` instead of aborting the sweep.

In [ ]:
ENERGY_COLUMNS = ['hour', 'renewable_available', 'grid_price', 'pue', 'baseline_load']


def proxy_model_config(clusters: pd.DataFrame, energy: pd.DataFrame) -> ModelConfig:
    # Copied from notebook 21: an energy-only MILP aligned with QUBO prices.
    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * 10.0,
        renewable_price=1_000_000.0,
        peak_price=0.0,
        delta_t=DELTA_T,
        pue=float(energy['pue'].iloc[0]),
    )


def full_model_config(clusters: pd.DataFrame, energy: pd.DataFrame) -> ModelConfig:
    # Same parameters as solve_one in scripts/run_long_duration_focused_sweep.py.
    pue = float(energy['pue'].iloc[0])
    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * pue * CONTRACTED_POWER_RATIO,
        renewable_price=float(energy['renewable_price'].iloc[0]),
        peak_price=PEAK_PRICE,
        delta_t=DELTA_T,
        pue=pue,
    )


def _solve_with_diagnostics(model: gp.Model, time_limit: float) -> dict:
    model.Params.OutputFlag = 0
    model.Params.Threads = THREADS
    model.update()  # relax() copies only the applied model state

    relaxed = model.relax()
    relaxed.Params.OutputFlag = 0
    relaxed.Params.Threads = THREADS
    relaxed.optimize()
    lp_objective = float(relaxed.ObjVal) if relaxed.Status == GRB.OPTIMAL else None

    first_incumbent = {'runtime': None}

    def on_mipsol(callback_model, where):
        if where == GRB.Callback.MIPSOL and first_incumbent['runtime'] is None:
            first_incumbent['runtime'] = float(callback_model.cbGet(GRB.Callback.RUNTIME))

    started = time.perf_counter()
    solve_model(model, time_limit=time_limit, mip_gap=MIP_GAP, callback=on_mipsol, raise_on_no_solution=False)
    runtime_s = time.perf_counter() - started
    has_solution = model.SolCount > 0
    try:
        best_bound = float(model.ObjBound)
    except gp.GurobiError:
        best_bound = None
    return {
        'status': int(model.Status),
        'has_solution': bool(has_solution),
        'runtime_s': runtime_s,
        'time_to_first_incumbent_s': first_incumbent['runtime'],
        'objective': float(model.ObjVal) if has_solution else None,
        'best_bound': best_bound,
        'mip_gap': float(model.MIPGap) if has_solution else None,
        'node_count': float(model.NodeCount),
        'lp_relaxation': lp_objective,
        'constraints': int(model.NumConstrs),
        'time_limit_s': float(time_limit),
    }


def solve_instance(jobs: pd.DataFrame, clusters: pd.DataFrame, energy: pd.DataFrame,
                   time_limit: float, *, solve_full: bool = True) -> dict:
    # Proxy MILP: exactly the problem the QUBO encodes.
    proxy_model, proxy_vars = build_milp_model(
        jobs, make_effective_price_qubo_inputs(energy), clusters, proxy_model_config(clusters, energy),
        model_name='qubo_aligned_proxy_milp',
        enforce_gpu_constraints=True, enforce_cpu_constraints=False, enforce_memory_constraints=False,
    )
    result = {'assignment_variables': len(proxy_vars['x']),
              'proxy': _solve_with_diagnostics(proxy_model, time_limit), 'full': None}
    if solve_full:
        full_model, _ = build_milp_model(
            jobs, energy[ENERGY_COLUMNS], clusters, full_model_config(clusters, energy),
            model_name='full_milp',
            enforce_gpu_constraints=True, enforce_cpu_constraints=False, enforce_memory_constraints=False,
        )
        result['full'] = _solve_with_diagnostics(full_model, time_limit)
    return result

## 4. Outcome labels and hardness score

Each solve receives one label:

| Label | Rule |
|---|---|
| `feasibility_hard` | no incumbent at the time limit |
| `optimality_hard` | incumbent found, not proven optimal, gap > 0.5 % at the time limit |
| `solved` | otherwise (optimal, or time-limited with gap ≤ 0.5 %) |

Hardness score used for ranking (higher = harder): proxy runtime if proven optimal; `time_limit × (1 + gap)` if stopped at the limit with an incumbent; `2 × time_limit` without an incumbent. The score is continuous across the three cases, so a time-limited instance always ranks above an instance solved before the limit.

The cell then solves the baseline (seed 7602) with both models at the screening time limit.

In [ ]:
def outcome_label(result: dict | None) -> str | None:
    if result is None:
        return None
    if not result['has_solution']:
        return 'feasibility_hard'
    if result['status'] != GRB.OPTIMAL and (result['mip_gap'] or 0.0) > HARD_GAP_THRESHOLD:
        return 'optimality_hard'
    return 'solved'


def hardness_score(result: dict | None) -> float | None:
    if result is None:
        return None
    limit = result['time_limit_s']
    if not result['has_solution']:
        return 2.0 * limit
    if result['status'] == GRB.OPTIMAL:
        return result['runtime_s']
    return limit * (1.0 + result['mip_gap'])


baseline_solve = solve_instance(baseline_jobs, baseline_clusters, baseline_energy, TIME_LIMIT_SCREEN)
baseline_table = pd.DataFrame([
    {'model': name, **baseline_solve[name], 'outcome_label': outcome_label(baseline_solve[name]),
     'hardness_score': hardness_score(baseline_solve[name])}
    for name in ('proxy', 'full')
])
baseline_table['lp_root_gap'] = [
    lp_root_gap(row.objective, row.lp_relaxation) for row in baseline_table.itertuples()
]
display(baseline_table)
BASELINE_PROXY_SCORE = hardness_score(baseline_solve['proxy'])
print(f'Baseline proxy hardness score: {BASELINE_PROXY_SCORE:.2f} '
      f'(assignment variables: {baseline_solve["assignment_variables"]})')

### Profiles, generation and caching

A **profile** is one generator configuration plus one price shape. Its descriptive name encodes every non-default factor, for example `u094_win20-28_sizesB_jitter`.

`num_jobs` comes from `jobs_for_target_utilization`, so the utilisation accept/reject loop starts at the right job count. The tolerance is tightened to ±0.02 (the original sweep used ±0.06, which would make the 0.82 and 0.88 levels overlap).

Note on high utilisation: in a probe run before this notebook, almost every attempt at U = 0.94 that passed the utilisation check was rejected at the **planting** step, because the greedy certificate cannot pack the jobs. Accepted instances therefore land at the low end of the tolerance band. Every row records the realised utilisation and the generator's rejection counts; failed generations are cached and logged, not retried.

Instance folders are written in notebook 21's format during the sweep, so export is a folder copy.

In [ ]:
PROFILE_REGISTRY: dict[str, dict] = {}
DAY_ENERGY_CACHE: dict[str, pd.DataFrame] = {}
PRICE_SUFFIX = {'none': '', 'two_valleys': '_twovalleys', 'jitter': '_jitter'}


class GenerationFailed(RuntimeError):
    pass


def make_profile(util, window, sizes, *, day='base', price_shape='none', power_intensity=(1.0, 1.0),
                 steer=True, planting='greedy_min_peak', variant='baseline') -> dict:
    profile = {'util': float(util), 'window': tuple(window), 'sizes': sizes, 'day': day,
               'price_shape': price_shape, 'power_intensity': tuple(power_intensity),
               'steer': bool(steer), 'planting': planting, 'variant': variant}
    PROFILE_REGISTRY.setdefault(profile_name(profile), profile)
    return profile


def profile_name(profile: dict) -> str:
    name = f"u{round(profile['util'] * 100):03d}_win{profile['window'][0]}-{profile['window'][1]}_sizes{profile['sizes']}"
    if profile['day'] != 'base':
        name += f"_day-{profile['day'].replace('_', '')}"
    name += PRICE_SUFFIX[profile['price_shape']]
    if profile['power_intensity'] != (1.0, 1.0):
        name += f"_pow{round(profile['power_intensity'][0] * 100):03d}"
    if not profile['steer']:
        name += '_nosteer'
    if profile['planting'] != 'greedy_min_peak':
        name += '_randplant'
    return name


def build_config(profile: dict) -> SingleClusterStressConfig:
    values, probabilities = SIZE_SETS[profile['sizes']]
    config = SingleClusterStressConfig(
        target_utilization=profile['util'],
        utilization_tolerance=UTILIZATION_TOLERANCE,
        node_demand_values=values,
        node_demand_probabilities=probabilities,
        duration_values=DURATION_VALUES,
        duration_probabilities=DURATION_PROBABILITIES,
        min_start_options=profile['window'][0],
        max_start_options=profile['window'][1],
        min_price_spread_ratio=1.15,
        # Without steering the cheap-slot pressure check is switched off too,
        # so the factor is isolated rather than replaced by rejection sampling.
        min_peak_potential_load_ratio=1.50 if profile['steer'] else 0.0,
        max_attempts=MAX_GENERATION_ATTEMPTS,
        steer_to_cheap_slot=profile['steer'],
        power_intensity_range=profile['power_intensity'],
        planting=profile['planting'],
    )
    return replace(config, num_jobs=jobs_for_target_utilization(config, profile['util']))


def profile_energy(profile: dict, seed: int) -> pd.DataFrame:
    day = profile['day']
    if day not in DAY_ENERGY_CACHE:
        DAY_ENERGY_CACHE[day] = build_energy_for_day(day)
    energy = DAY_ENERGY_CACHE[day]
    if profile['price_shape'] == 'two_valleys':
        return apply_two_valleys(energy, TWO_VALLEY_NIGHT_SLOTS, TWO_VALLEY_DEPTH)
    if profile['price_shape'] == 'jitter':
        return apply_price_jitter(energy, JITTER_REL_STD, seed)
    return energy


def price_shape_record(profile: dict, seed: int) -> dict:
    shape = profile['price_shape']
    params = {}
    if shape == 'two_valleys':
        params = {'night_slots': list(TWO_VALLEY_NIGHT_SLOTS), 'depth_quantile': TWO_VALLEY_DEPTH}
    elif shape == 'jitter':
        params = {'rel_std': JITTER_REL_STD, 'seed': seed}
    return {'name': shape, 'params': params}


def instance_dir(stage: str, profile: dict, seed: int) -> Path:
    return RESULTS_ROOT / 'instances' / stage / f'{profile_name(profile)}_seed_{seed}'


def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray, tuple, set)):
        return list(value)
    raise TypeError(f'not JSON serialisable: {type(value)}')


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, default=_json_default))


def materialise_instance(profile: dict, seed: int, out_dir: Path):
    # Generate once and persist in notebook 21's format; reload on later calls.
    if (out_dir / 'jobs.csv').exists():
        return (pd.read_csv(out_dir / 'jobs.csv'), pd.read_csv(out_dir / 'clusters.csv'),
                pd.read_csv(out_dir / 'energy_inputs_15m.csv'),
                json.loads((out_dir / 'generation_metadata.json').read_text()))
    error_file = out_dir / 'generation_error.txt'
    if error_file.exists():
        raise GenerationFailed(error_file.read_text())

    config = build_config(profile)
    name = out_dir.name
    started = time.perf_counter()
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        jobs, clusters, energy, hidden, diagnostics, metadata = generate_single_cluster_stress_instance(
            profile_energy(profile, seed), config, seed, name
        )
    except RuntimeError as error:
        error_file.write_text(str(error))
        raise GenerationFailed(str(error)) from error
    metadata.update({
        'profile_name': profile_name(profile),
        'profile': {key: value for key, value in profile.items() if key != 'variant'},
        'price_shape': price_shape_record(profile, seed),
        'energy_day': profile['day'],
        'energy_date': ENERGY_SCENARIOS[profile['day']]['date'],
        'generation_seconds': time.perf_counter() - started,
    })
    jobs.to_csv(out_dir / 'jobs.csv', index=False)
    clusters.to_csv(out_dir / 'clusters.csv', index=False)
    energy.to_csv(out_dir / 'energy_inputs_15m.csv', index=False)
    hidden.to_csv(out_dir / 'hidden_feasibility_certificate.csv', index=False)
    diagnostics.to_csv(out_dir / 'generation_diagnostics.csv', index=False)
    write_json(out_dir / 'generation_metadata.json', metadata)
    return jobs, clusters, energy, metadata

### Running one instance

`run_instance` generates (or reloads) one instance, enforces the size budget, solves it and writes `gurobi_summary.csv`. The summary has **the same columns `solve_one` writes, in the same order**, followed by `proxy_status`, `proxy_runtime_s`, `proxy_mip_gap`, `proxy_objective`, `proxy_best_bound`, `outcome_label` (proxy MILP) and further diagnostics. Hardness predictors are prefixed `m_` and also saved to `hardness_metrics.json`.

When the full MILP is not solved (Stage 1 screening), its `solve_one` columns are empty.

In [ ]:
SOLVE_ONE_COLUMNS = [
    'instance', 'profile', 'seed', 'status', 'has_solution', 'runtime_s', 'mip_gap', 'objective',
    'best_bound', 'assignment_variables', 'constraints', 'contracted_power_ratio', 'utilization',
    'mean_duration_slots', 'mean_node_demand', 'mean_start_options', 'mean_price_spread_ratio',
    'cheap_peak_potential_load_ratio', 'hidden_peak_load_ratio', 'config',
]
PREDICTORS = [
    'utilization', 'max_rho', 'mean_rho', 'slots_rho_gt_1', 'rho_at_cheapest', 'omega',
    'dmax_over_c', 'demand_gcd', 'distinct_job_types', 'savings_weight_corr', 'price_tie_share',
    'mean_start_options', 'assignment_variables', 'lp_root_gap',
]


def summary_row(stage, profile, seed, metadata, metrics, solved) -> dict:
    proxy, full = solved['proxy'], solved['full']
    full = full or {}
    row = {
        'instance': f'{profile_name(profile)}_seed_{seed}',
        'profile': profile_name(profile),
        'seed': seed,
        'status': full.get('status'),
        'has_solution': full.get('has_solution'),
        'runtime_s': full.get('runtime_s'),
        'mip_gap': full.get('mip_gap'),
        'objective': full.get('objective'),
        'best_bound': full.get('best_bound'),
        'assignment_variables': solved['assignment_variables'],
        'constraints': full.get('constraints'),
        'contracted_power_ratio': CONTRACTED_POWER_RATIO,
        'utilization': metadata['total_node_slot_utilization'],
        'mean_duration_slots': metadata['mean_duration_slots'],
        'mean_node_demand': metadata['mean_node_demand'],
        'mean_start_options': metadata['mean_start_options'],
        'mean_price_spread_ratio': metadata['mean_price_spread_ratio'],
        'cheap_peak_potential_load_ratio': metadata['cheap_peak_potential_load_ratio'],
        'hidden_peak_load_ratio': metadata['hidden_peak_load_ratio'],
        'config': json.dumps(metadata['config']),
        # Proxy MILP: the ranking model.
        'proxy_status': proxy['status'],
        'proxy_runtime_s': proxy['runtime_s'],
        'proxy_mip_gap': proxy['mip_gap'],
        'proxy_objective': proxy['objective'],
        'proxy_best_bound': proxy['best_bound'],
        'outcome_label': outcome_label(proxy),
        # Further diagnostics.
        'hardness_score': hardness_score(proxy),
        'proxy_has_solution': proxy['has_solution'],
        'proxy_time_to_first_incumbent_s': proxy['time_to_first_incumbent_s'],
        'proxy_node_count': proxy['node_count'],
        'proxy_lp_relaxation': proxy['lp_relaxation'],
        'proxy_constraints': proxy['constraints'],
        'full_outcome_label': outcome_label(solved['full']),
        'full_hardness_score': hardness_score(solved['full']),
        'full_time_to_first_incumbent_s': full.get('time_to_first_incumbent_s'),
        'full_node_count': full.get('node_count'),
        'full_lp_relaxation': full.get('lp_relaxation'),
        'stage': stage,
        'variant': profile['variant'],
        'time_limit_s': proxy['time_limit_s'],
        **SOLVER_LOG,
        'generation_attempt': metadata['attempt'],
        'generation_rejections': json.dumps(metadata.get('rejections', {})),
        'profile_spec': json.dumps({key: value for key, value in profile.items() if key != 'variant'},
                                   default=_json_default),
    }
    row.update({f'm_{key}': value for key, value in metrics.items()})
    return row


def run_instance(profile: dict, seed: int, stage: str, time_limit: float, *, solve_full: bool) -> dict:
    out_dir = instance_dir(stage, profile, seed)
    summary_path = out_dir / 'gurobi_summary.csv'
    identity = {'instance': f'{profile_name(profile)}_seed_{seed}', 'profile': profile_name(profile),
                'seed': seed, 'stage': stage, 'variant': profile['variant']}
    if summary_path.exists():
        return {**pd.read_csv(summary_path).iloc[0].to_dict(), 'variant': profile['variant']}

    config = build_config(profile)
    if config.num_jobs * config.min_start_options > ASSIGNMENT_BUDGET:
        return {**identity, 'skip_reason': 'size_budget_predicted',
                'assignment_variables': config.num_jobs * config.min_start_options}
    try:
        jobs, clusters, energy, metadata = materialise_instance(profile, seed, out_dir)
    except GenerationFailed as error:
        return {**identity, 'skip_reason': 'generation_failed', 'skip_detail': str(error)[-200:]}

    metrics = compute_hardness_metrics(jobs, energy, clusters, delta_t=DELTA_T)
    if metrics['assignment_variables'] > ASSIGNMENT_BUDGET:
        return {**identity, 'skip_reason': 'size_budget', 'assignment_variables': metrics['assignment_variables']}

    solved = solve_instance(jobs, clusters, energy, time_limit, solve_full=solve_full)
    metrics['lp_root_gap'] = lp_root_gap(solved['proxy']['objective'], solved['proxy']['lp_relaxation'])
    write_json(out_dir / 'hardness_metrics.json', metrics)
    row = summary_row(stage, profile, seed, metadata, metrics, solved)
    pd.DataFrame([row]).to_csv(summary_path, index=False)
    return row


def run_grid(profiles: list[dict], seeds, stage: str, time_limit: float, *, solve_full: bool) -> pd.DataFrame:
    rows = []
    for profile in profiles:
        for seed in seeds:
            started = time.perf_counter()
            row = run_instance(profile, seed, stage, time_limit, solve_full=solve_full)
            rows.append(row)
            status = row.get('outcome_label') if isinstance(row.get('outcome_label'), str) else row.get('skip_reason')
            extra = (f"proxy {row['proxy_runtime_s']:.1f}s gap {row['proxy_mip_gap'] or 0:.4f}"
                     if 'proxy_runtime_s' in row and pd.notna(row.get('proxy_runtime_s')) else '')
            print(f'[{stage}] {profile_name(profile)} seed {seed}: {status} {extra} '
                  f'({time.perf_counter() - started:.1f}s)', flush=True)
            pd.DataFrame(rows).to_csv(RESULTS_ROOT / f'{stage}_partial.csv', index=False)
    frame = pd.DataFrame(rows)
    frame.to_csv(RESULTS_ROOT / f'{stage}.csv', index=False)
    return frame


def rank_profiles(frame: pd.DataFrame, min_seeds: int = 2) -> pd.DataFrame:
    if frame.empty or 'outcome_label' not in frame:
        return pd.DataFrame(columns=['profile', 'median_score'])
    solved = frame[frame['outcome_label'].isin(['solved', 'optimality_hard', 'feasibility_hard'])]
    if solved.empty:
        return pd.DataFrame(columns=['profile', 'median_score'])
    ranking = solved.groupby('profile').agg(
        median_score=('hardness_score', 'median'),
        seeds=('seed', 'nunique'),
        hard_seeds=('outcome_label', lambda labels: int((labels != 'solved').sum())),
        median_proxy_runtime_s=('proxy_runtime_s', 'median'),
        median_proxy_gap=('proxy_mip_gap', 'median'),
        assignment_variables=('assignment_variables', 'max'),
    ).reset_index()
    ranking = ranking[ranking['seeds'] >= min(min_seeds, ranking['seeds'].max())]
    return ranking.sort_values('median_score', ascending=False).reset_index(drop=True)

## 5. Stage 1a: screening grid

Base price day and default generator flags; utilisation × start-window width × node-size set, 3 seeds each at 60 s, **proxy MILP only**.

| Factor | Levels |
|---|---|
| `target_utilization` | 0.82, 0.88, 0.94 |
| Start options (min, max) | (8, 16), (12, 20), (20, 28) |
| Node sizes / probabilities | A = (16, 32, 64, 96) / (.15, .35, .35, .15); B = (48, 80, 112) / (.4, .4, .2); C = (32, 64, 96, 128) / (.25, .35, .25, .15) |

Configurations above the 1,000-variable budget are skipped and logged. Size set A with (20, 28) windows exceeds the budget at every utilisation.

In [ ]:
stage1a_profiles = [
    make_profile(util, window, sizes)
    for util in UTILIZATION_LEVELS for window in WINDOW_LEVELS for sizes in SIZE_SETS
]
stage1a_path = RESULTS_ROOT / 'stage1a.csv'
if RUN_SCREENING:
    stage1a = run_grid(stage1a_profiles, SCREEN_SEEDS, 'stage1a', TIME_LIMIT_SCREEN, solve_full=False)
elif stage1a_path.exists():
    stage1a = pd.read_csv(stage1a_path)
else:
    stage1a = pd.DataFrame()

if not stage1a.empty:
    if 'skip_reason' in stage1a:
        display(stage1a[stage1a['skip_reason'].notna()].groupby(['profile', 'skip_reason']).size()
                .rename('skipped_seeds').reset_index())
    stage1a_ranking = rank_profiles(stage1a)
    display(stage1a_ranking)
else:
    stage1a_ranking = rank_profiles(stage1a)

## 6. Stage 1b: price and savings structure

One-factor-at-a-time changes from each of the 3 hardest Stage 1a configurations (3 seeds each, 60 s, proxy only):

| Factor | Levels tested (baseline in bold) |
|---|---|
| Price | **base day**; the other two `ENERGY_SCENARIOS` days; two valleys; jitter (rel. std 0.01) |
| `power_intensity_range` | **(1.0, 1.0)**, (0.85, 1.0), (0.5, 1.0) |
| `steer_to_cheap_slot` | **True**, False (cheap-slot pressure check also off) |
| `planting` | **greedy_min_peak**, random_feasible |

Why these factors: on seed 7602 the per-job saving correlates 0.68 with node-slots, so a greedy "largest jobs into the cheapest slots" rule is nearly optimal. Power intensity breaks that proportionality; a second price valley and other days change where the savings sit; and random planting removes a certificate that is itself a load-balancing heuristic.

In [ ]:
def oat_variants(base: dict) -> list[dict]:
    common = {key: base[key] for key in ('util', 'window', 'sizes')}
    variants = [
        make_profile(**common, day=day, variant=f'price:day-{day}')
        for day in ENERGY_SCENARIOS if day != 'base'
    ]
    variants += [
        make_profile(**common, price_shape='two_valleys', variant='price:two_valleys'),
        make_profile(**common, price_shape='jitter', variant='price:jitter'),
        make_profile(**common, power_intensity=(0.85, 1.0), variant='power:0.85-1.0'),
        make_profile(**common, power_intensity=(0.5, 1.0), variant='power:0.5-1.0'),
        make_profile(**common, steer=False, variant='steer:False'),
        make_profile(**common, planting='random_feasible', variant='planting:random_feasible'),
    ]
    return variants


stage1b_bases = [PROFILE_REGISTRY[name] for name in stage1a_ranking['profile'].head(TOP_STAGE1A_FOR_STAGE1B)]
stage1b_profiles = [variant for base in stage1b_bases for variant in oat_variants(base)]
print('Stage 1b base configurations:', [profile_name(base) for base in stage1b_bases])

stage1b_path = RESULTS_ROOT / 'stage1b.csv'
if RUN_PRICE_STRUCTURE and stage1b_profiles:
    stage1b = run_grid(stage1b_profiles, SCREEN_SEEDS, 'stage1b', TIME_LIMIT_SCREEN, solve_full=False)
elif stage1b_path.exists():
    stage1b = pd.read_csv(stage1b_path)
else:
    stage1b = pd.DataFrame()

screening = pd.concat([stage1a, stage1b], ignore_index=True)
screening_ranking = rank_profiles(screening)
display(screening_ranking.head(15))

## 7. Stage 2: confirmation

The top 5 profiles by median Stage 1 hardness score are re-generated on 5 **new** seeds and solved for 300 s with both the proxy and the full MILP. A profile is kept only if at least 3 of the 5 seeds are `optimality_hard` or `feasibility_hard` **on the proxy MILP**.

This stage is behind `RUN_CONFIRMATION` because its worst case is about 5 × 5 × (300 s + 300 s) ≈ 4 hours. Cached results are reused when the flag is off.

In [ ]:
confirmation_candidates = [PROFILE_REGISTRY[name] for name in screening_ranking['profile'].head(TOP_FOR_CONFIRMATION)]
print('Confirmation candidates:', [profile_name(profile) for profile in confirmation_candidates])

confirm_path = RESULTS_ROOT / 'confirm.csv'
if RUN_CONFIRMATION and confirmation_candidates:
    confirmation = run_grid(confirmation_candidates, CONFIRM_SEEDS, 'confirm', TIME_LIMIT_CONFIRM, solve_full=True)
elif confirm_path.exists():
    confirmation = pd.read_csv(confirm_path)
else:
    confirmation = pd.DataFrame()

if not confirmation.empty and 'outcome_label' in confirmation:
    confirmation_ranking = rank_profiles(confirmation, min_seeds=1)
    confirmation_ranking['kept'] = confirmation_ranking['hard_seeds'] >= MIN_HARD_SEEDS
    confirmation_ranking['harder_than_baseline'] = confirmation_ranking['median_score'] > BASELINE_PROXY_SCORE
    display(confirmation_ranking)
    kept_profiles = confirmation_ranking.loc[confirmation_ranking['kept'], 'profile'].tolist()
    display(confirmation.groupby(['profile', 'outcome_label', 'full_outcome_label']).size()
            .rename('seeds').reset_index())
else:
    confirmation_ranking = pd.DataFrame()
    kept_profiles = []
    print('No confirmation results yet: set RUN_CONFIRMATION = True and re-run this cell.')
print('Kept profiles:', kept_profiles)

## 8. Analysis

Figures are saved to `results/hard_frontier/figures/`.

1. Heat map of the median proxy-MILP hardness score over utilisation × start-window level, one panel per size set (Stage 1a). Grey cells were skipped (size budget or generation failure).
2. Each structural predictor against log₁₀ proxy runtime, coloured by outcome label (Stage 1a + 1b).
3. Outcome labels per Stage 1b factor level, with the Stage 1a baseline of the same base configurations.
4. Spearman rank correlation of each predictor with the hardness score.

In [ ]:
LABEL_COLORS = {'solved': '#4c78a8', 'optimality_hard': '#f58518', 'feasibility_hard': '#e45756'}


def solved_rows(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty or 'outcome_label' not in frame:
        return frame.iloc[0:0]
    return frame[frame['outcome_label'].isin(LABEL_COLORS)].copy()


stage1a_solved = solved_rows(stage1a)
if not stage1a_solved.empty:
    specs = stage1a_solved['profile'].map(PROFILE_REGISTRY)
    stage1a_solved['util_level'] = specs.map(lambda spec: spec['util'])
    stage1a_solved['window_level'] = specs.map(lambda spec: f"{spec['window'][0]}-{spec['window'][1]}")
    stage1a_solved['size_set'] = specs.map(lambda spec: spec['sizes'])
    window_labels = [f'{low}-{high}' for low, high in WINDOW_LEVELS]
    vmax = stage1a_solved['hardness_score'].max()
    figure, axes = plt.subplots(1, len(SIZE_SETS), figsize=(4.6 * len(SIZE_SETS), 3.8), squeeze=False)
    for axis, size_set in zip(axes[0], SIZE_SETS):
        grid = (stage1a_solved[stage1a_solved['size_set'] == size_set]
                .pivot_table(index='util_level', columns='window_level', values='hardness_score', aggfunc='median')
                .reindex(index=list(UTILIZATION_LEVELS), columns=window_labels))
        axis.set_facecolor('#d9d9d9')
        image = axis.imshow(np.ma.masked_invalid(grid.to_numpy(dtype=float)), cmap='viridis',
                            vmin=0, vmax=vmax, origin='lower', aspect='auto')
        for (row, col), value in np.ndenumerate(grid.to_numpy(dtype=float)):
            axis.text(col, row, 'skip' if np.isnan(value) else f'{value:.0f}', ha='center', va='center',
                      color='black' if np.isnan(value) or value > 0.6 * vmax else 'white', fontsize=9)
        axis.set_xticks(range(len(window_labels)), window_labels)
        axis.set_yticks(range(len(UTILIZATION_LEVELS)), UTILIZATION_LEVELS)
        axis.set_xlabel('start options (min-max)')
        axis.set_ylabel('target utilisation')
        axis.set_title(f'size set {size_set}')
    figure.colorbar(image, ax=axes[0].tolist(), label='median proxy hardness score (s)')
    figure.suptitle(f'Stage 1a: proxy-MILP hardness (time limit {TIME_LIMIT_SCREEN}s)')
    figure.savefig(FIGURE_ROOT / 'stage1a_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No Stage 1a results to plot.')

In [ ]:
screening_solved = solved_rows(screening)
if not screening_solved.empty:
    columns = 4
    rows = math.ceil(len(PREDICTORS) / columns)
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.1 * rows), squeeze=False)
    log_runtime = np.log10(screening_solved['proxy_runtime_s'].astype(float).clip(lower=1e-2))
    for axis, predictor in zip(axes.flat, PREDICTORS):
        values = screening_solved[f'm_{predictor}'].astype(float)
        for label, color in LABEL_COLORS.items():
            mask = screening_solved['outcome_label'] == label
            axis.scatter(values[mask], log_runtime[mask], s=18, alpha=0.8, color=color, label=label)
        axis.set_xlabel(predictor)
        axis.set_ylabel('log10 proxy runtime (s)')
    for axis in list(axes.flat)[len(PREDICTORS):]:
        axis.axis('off')
    axes.flat[0].legend(fontsize=8)
    figure.suptitle('Structural predictors vs proxy-MILP runtime (Stage 1a + 1b)')
    figure.tight_layout()
    figure.savefig(FIGURE_ROOT / 'predictor_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
stage1b_solved = solved_rows(stage1b)
if not stage1b_solved.empty:
    base_names = [profile_name(base) for base in stage1b_bases]
    reference = solved_rows(stage1a)
    reference = reference[reference['profile'].isin(base_names)].assign(variant='baseline')
    factor_rows = pd.concat([reference, stage1b_solved], ignore_index=True)
    counts = (factor_rows.groupby(['variant', 'outcome_label']).size().unstack(fill_value=0)
              .reindex(columns=list(LABEL_COLORS), fill_value=0))
    order = ['baseline'] + sorted(level for level in counts.index if level != 'baseline')
    counts = counts.reindex(order)
    axis = counts.plot(kind='bar', stacked=True, figsize=(11, 4),
                       color=[LABEL_COLORS[label] for label in counts.columns])
    axis.set_ylabel('instances')
    axis.set_xlabel('Stage 1b factor level')
    axis.set_title('Proxy-MILP outcome labels per factor level')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / 'stage1b_outcomes.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(factor_rows.groupby('variant').agg(
        instances=('seed', 'size'), median_score=('hardness_score', 'median'),
        median_runtime_s=('proxy_runtime_s', 'median'), median_gap=('proxy_mip_gap', 'median'),
    ).reindex(order))

In [ ]:
analysis_rows = pd.concat([screening_solved, solved_rows(confirmation)], ignore_index=True)
if len(analysis_rows) >= 3:
    import warnings
    from scipy.stats import ConstantInputWarning
    warnings.simplefilter('ignore', ConstantInputWarning)  # constant predictors (e.g. demand_gcd) give NaN
    spearman = pd.DataFrame([
        {'predictor': predictor,
         'spearman_rho': analysis_rows[f'm_{predictor}'].astype(float)
             .corr(analysis_rows['hardness_score'].astype(float), method='spearman'),
         'non_missing': int(analysis_rows[f'm_{predictor}'].notna().sum())}
        for predictor in PREDICTORS
    ])
    spearman['abs_rho'] = spearman['spearman_rho'].abs()
    spearman = spearman.sort_values('abs_rho', ascending=False).drop(columns='abs_rho').reset_index(drop=True)
    spearman.to_csv(RESULTS_ROOT / 'predictor_spearman.csv', index=False)
    display(spearman)
else:
    spearman = pd.DataFrame(columns=['predictor', 'spearman_rho'])
    print('Not enough solved instances for correlations.')

### Automatic summary

The next cell prints the facts for the written summary: the hardest conditions found, the predictors most correlated with difficulty, and the split between feasibility-hard and optimality-hard instances.

In [ ]:
def describe(frame: pd.DataFrame, title: str) -> str:
    solved = solved_rows(frame)
    if solved.empty:
        return f'- {title}: no solved instances.'
    split = solved['outcome_label'].value_counts().reindex(list(LABEL_COLORS), fill_value=0)
    return (f'- {title}: {len(solved)} solved instances; '
            + ', '.join(f'{label} = {count}' for label, count in split.items()))


lines = ['**Outcome split (proxy MILP)**', describe(stage1a, 'Stage 1a'), describe(stage1b, 'Stage 1b'),
         describe(confirmation, 'Stage 2')]
if 'skip_reason' in screening:
    skipped = screening['skip_reason'].value_counts()
    lines.append('- Skipped in Stage 1: ' + (', '.join(f'{reason} = {count}' for reason, count in skipped.items())
                                            or 'none'))
lines += ['', f'**Baseline** `long_wide_window` seed 7602 proxy hardness score: {BASELINE_PROXY_SCORE:.1f}', '',
          '**Hardest Stage 1 profiles (median proxy hardness score)**']
for row in screening_ranking.head(5).itertuples():
    lines.append(f'- `{row.profile}`: score {row.median_score:.1f}, hard seeds {row.hard_seeds}/{row.seeds}, '
                 f'median gap {0 if pd.isna(row.median_proxy_gap) else row.median_proxy_gap:.4f}, '
                 f'{int(row.assignment_variables)} assignment variables')
if not confirmation_ranking.empty:
    lines += ['', '**Stage 2**']
    for row in confirmation_ranking.itertuples():
        lines.append(f'- `{row.profile}`: hard seeds {row.hard_seeds}/{row.seeds}, score {row.median_score:.1f}, '
                     f"{'kept' if row.kept else 'rejected'}, "
                     f"{'harder' if row.harder_than_baseline else 'not harder'} than baseline")
if not spearman.empty:
    lines += ['', '**Strongest predictors (Spearman rho with hardness score)**']
    for row in spearman.head(5).itertuples():
        lines.append(f'- `{row.predictor}`: {row.spearman_rho:+.2f}')
if not kept_profiles:
    lines += ['', '**No profile was confirmed harder than the baseline on the proxy MILP within this grid.**']
display(Markdown('\n'.join(lines)))

### Summary (fill in after the full run)

*Write this cell after running Stages 1a, 1b and 2 with the production settings, using the automatic summary above.*

- **Hardest conditions found:** …
- **Predictors that explain difficulty:** …
- **Feasibility-hard vs optimality-hard split:** …
- **Generator limits observed:** at U = 0.94 the planted certificate usually fails; record how many configurations were skipped for generation failure or for the size budget.

## 9. Export

For each kept profile, the confirmation seeds are sorted by proxy hardness score: the **two least hard** become development seeds, and the **hardest** becomes the held-out target. Their folders are copied unchanged to `data/instances/single_cluster_15m/hard_frontier/<profile>_seed_<seed>/`. Each folder contains `jobs.csv`, `clusters.csv`, `energy_inputs_15m.csv` (after any price-shape helper, with `effective_price`), `hidden_feasibility_certificate.csv`, `generation_diagnostics.csv`, `generation_metadata.json` (with `price_shape` and the scenario day), `gurobi_summary.csv` and `hardness_metrics.json`.

`manifest.csv` lists every exported instance. Each folder is then loaded with a verbatim copy of notebook 21's `load_instance` to prove compatibility.

In [ ]:
REQUIRED_FILES = [
    'jobs.csv', 'clusters.csv', 'energy_inputs_15m.csv', 'hidden_feasibility_certificate.csv',
    'generation_diagnostics.csv', 'generation_metadata.json', 'gurobi_summary.csv', 'hardness_metrics.json',
]
exported_profiles: dict[str, dict] = {}
manifest_rows = []

for name in kept_profiles:
    profile = PROFILE_REGISTRY[name]
    seeds = solved_rows(confirmation[confirmation['profile'] == name]).sort_values('hardness_score')
    if len(seeds) < 3:
        print(f'{name}: fewer than 3 solved confirmation seeds; not exported.')
        continue
    roles = [(seeds.iloc[0], 'development'), (seeds.iloc[1], 'development'), (seeds.iloc[-1], 'target')]
    for row, role in roles:
        seed = int(row['seed'])
        source = instance_dir('confirm', profile, seed)
        destination = EXPORT_ROOT / f'{name}_seed_{seed}'
        missing = [file for file in REQUIRED_FILES if not (source / file).exists()]
        assert not missing, f'{source} is missing {missing}'
        destination.mkdir(parents=True, exist_ok=True)
        for file in REQUIRED_FILES:
            shutil.copy2(source / file, destination / file)
        manifest_rows.append({
            'profile_name': name, 'seed': seed, 'role': role,
            'assignment_variables': int(row['assignment_variables']),
            'outcome_label': row['outcome_label'],
            'proxy_runtime_s': row['proxy_runtime_s'], 'proxy_mip_gap': row['proxy_mip_gap'],
            'full_mip_gap': row['mip_gap'], 'config': row['config'],
        })
    exported_profiles[name] = {
        'development_seeds': (int(seeds.iloc[0]['seed']), int(seeds.iloc[1]['seed'])),
        'target_seed': int(seeds.iloc[-1]['seed']),
    }

if manifest_rows:
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest_path = EXPORT_ROOT / 'manifest.csv'
    manifest = pd.DataFrame(manifest_rows)
    if manifest_path.exists():
        previous = pd.read_csv(manifest_path)
        manifest = (pd.concat([previous, manifest], ignore_index=True)
                    .drop_duplicates(['profile_name', 'seed'], keep='last'))
    manifest.to_csv(manifest_path, index=False)
    display(manifest)
else:
    print('Nothing exported.')

In [ ]:
# Verbatim copy of notebook 21's load_instance (cell 5). It reads the module-level
# INSTANCE_ROOT and PROFILE_NAME, which are set here exactly as notebook 21 would.
def load_instance(seed: int):
    # Load one persisted 15-minute one-cluster stress instance.
    instance_dir = INSTANCE_ROOT / f'{PROFILE_NAME}_seed_{seed}'
    jobs = pd.read_csv(instance_dir / 'jobs.csv')
    clusters = pd.read_csv(instance_dir / 'clusters.csv')
    energy = pd.read_csv(instance_dir / 'energy_inputs_15m.csv')
    gurobi = pd.read_csv(instance_dir / 'gurobi_summary.csv')
    return instance_dir, jobs, clusters, energy, gurobi


INSTANCE_ROOT = EXPORT_ROOT
for PROFILE_NAME, roles in exported_profiles.items():
    for seed in (*roles['development_seeds'], roles['target_seed']):
        _, jobs, clusters, energy, gurobi = load_instance(seed)
        assert 'effective_price' in energy and len(energy) == 96
        assert set(SOLVE_ONE_COLUMNS) <= set(gurobi.columns)
        qubo_inputs = make_effective_price_qubo_inputs(energy)
        assignment_variables = int((jobs['latest_start'] - jobs['earliest_start'] + 1).sum())
        assert assignment_variables <= ASSIGNMENT_BUDGET
        print(f'{PROFILE_NAME} seed {seed}: {len(jobs)} jobs, {assignment_variables} assignment variables, '
              f'proxy label {gurobi["outcome_label"].iloc[0]}; loads with notebook 21 load_instance.')

### Notebook 21 constants

To benchmark an exported profile in notebook 21, replace only these constants.

In [ ]:
if not exported_profiles:
    print('No exported profiles yet.')
for name, roles in exported_profiles.items():
    root = EXPORT_ROOT.relative_to(REPOSITORY_ROOT).as_posix()
    print(f"INSTANCE_ROOT = REPOSITORY_ROOT / '{root}'")
    print(f"PROFILE_NAME = '{name}'")
    print(f"DEVELOPMENT_SEEDS = {roles['development_seeds']}")
    print(f"TARGET_SEED = {roles['target_seed']}")
    print()